# FlowGuard - Setup & Validation
Mount Drive, install deps, validate raw data.

In [ ]:
import os
import shutil

# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 2. Clone / update the repo ────────────────────────────────────────────────
REPO_DIR = "/content/flowguard"
GITHUB_URL = "https://github.com/alperengoncu/guardian-engine-d3"

if not os.path.exists(REPO_DIR):
    !git clone "{GITHUB_URL}" "{REPO_DIR}"
else:
    !git -C "{REPO_DIR}" pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# ── 3. Copy datasets from Drive → data/raw/ ───────────────────────────────────
# Your datasets live directly inside /content/drive/MyDrive/flowguard/
# e.g. /content/drive/MyDrive/flowguard/NF-UNSW-NB15-v3.csv
DRIVE_DATA_DIR = "/content/drive/MyDrive/flowguard"
RAW_DIR = os.path.join(REPO_DIR, "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

EXPECTED_FILES = [
    "NF-UNSW-NB15-v3.csv",
    "NF-BoT-IoT-v3.csv",
    "NF-ToN-IoT-v3.csv",
    "NF-CICIDS2018-v3.csv",
]

for fname in EXPECTED_FILES:
    src = os.path.join(DRIVE_DATA_DIR, fname)
    dst = os.path.join(RAW_DIR, fname)
    if os.path.exists(dst):
        print(f"  [skip]  {fname} — already in data/raw/")
    elif os.path.exists(src):
        print(f"  [copy]  {fname} ({os.path.getsize(src)/1e9:.2f} GB) …", end=" ", flush=True)
        shutil.copy2(src, dst)
        print("done")
    else:
        print(f"  [MISSING] {fname} — not found in {DRIVE_DATA_DIR}")

print("\nSetup complete.")

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q flwr[simulation] shap umap-learn advertorch wandb pyarrow pyyaml tqdm plotly
!pip install -e . -q

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > GPU")

In [ ]:
from src.data.validate_raw import validate_raw_datasets

report = validate_raw_datasets(config_path="configs/base.yaml")
all_ok = all(info["exists"] for info in report.values())

if not all_ok:
    print("\nMISSING DATASETS:")
    for ds_name, info in report.items():
        if not info["exists"]:
            print(f"  -> {info['expected_file']}")
    print("\nDownload from: https://staff.itee.uq.edu.au/marius/NIDS_datasets/")
else:
    print("\nAll datasets found!")
    for ds_name, info in report.items():
        print(f"  {ds_name}: {info['rows']:,} rows, {info['cols']} cols, {info['size_mb']:.1f} MB")